In [8]:
import numpy as np
import pandas as pd
from scipy.stats import multivariate_normal, norm
from pysfa import SFA
from sklearn.linear_model import LassoCV

# Simulate data

In [7]:
# --- Parameters ---
n = 100         # sample size
nz = 200        # number of Z variables
beta = 0.5
sigv = 0.5
sigu = 1.2
ccy = 0.6
ccx = 0.8

# --- Covariance matrix for Z ---
Sigma = np.zeros((nz, nz))
for j in range(nz):
    for k in range(nz):
        Sigma[k, j] = 0.5 ** abs(j - k)

# --- δ_j = (1/j)^2 ---
j_seq = np.arange(1, nz + 1)
delta_j = (1 / j_seq) ** 2

# --- Derived quantities ---
true_sigma = sigv**2 + sigu**2
true_gamma = sigu**2 / true_sigma

# --- Generate X ~ N(0, Sigma) ---
Z = multivariate_normal.rvs(mean=np.zeros(nz), cov=Sigma, size=n)

# --- Generate v, u, and η ---
v = norm.rvs(scale=sigv, size=n)
u = np.abs(norm.rvs(scale=sigu, size=n))
eta = norm.rvs(size=n)
# --- True error ---
e = v - u

# --- Construct x  ---
X = Z @ delta_j * ccx + eta #Eq 12 d <- X%*%beta0*ccd+rnorm(n)
y = X * beta + Z @ delta_j * ccy + e #Eq 11 y <- d*alpha0+X%*%beta0*ccy+e

# Apply lasso (When you do rlasso(ytilde ~ Ztilde), LASSO only selects Zs that predict the part of y not explained by X.)

In [ ]:
# Ensure d is a column vector
X = X.reshape(-1, 1)  # shape (n, 1)

# Projection matrix onto X
H2 = X @ np.linalg.inv(X.T @ X) @ X.T   # equivalent to d %*% solve(t(d)%*%d) %*% t(d)

# Annihilator (residual-maker) matrix
M2 = np.eye(n) - H2

# Partial out y and X
ytilde = M2 @ y        # same as y - H2 @ y
Ztilde = M2 @ Z        # same as Z - H2 @ Z

# Now estimate true SFA 

In [15]:
# Combine d and X into one design matrix
W = np.column_stack((X, Z))

# Select first 8 columns of W
X_sfa = W[:, :8]   # d + first 7 Zs

# y must be 1D
y_sfa = y.reshape(-1)

# Initialize SFA model (production function, half-normal inefficiency)
sfa_model = SFA.SFA(y_sfa, X_sfa)
sfa_model.optimize()
# Extract coefficients

# Extract coefficients
beta_hat = sfa_model.get_beta()           # array: [β0, β1, ..., β7]
sigma_v2 = sfa_model.get_sigmav2()       # noise variance
sigma_u2 = sfa_model.get_sigmau2()       # inefficiency variance

# Convert variance → standard deviation
sigma_v = np.sqrt(sigma_v2)
sigma_u = np.sqrt(sigma_u2)

store_lm =  np.array([beta_hat[0], beta_hat[1], sigma_v, sigma_u])
store_lm

array([-0.18804735,  0.60686028,  0.48907736,  1.03729181])

## Now estimate over various 
## inclusion of Z variables
## Alternative 3, $\lambda_{BCCH}$

In [20]:
lasso_model = LassoCV(cv=5, fit_intercept=False, n_jobs=-1).fit(Ztilde, ytilde)

# Coefficients
lasso_coef = lasso_model.coef_

# Indices of selected variables (non-zero coefficients)
id_lasso = np.where(lasso_coef != 0)[0]

# Prepare Z for post-LASSO SFA
if len(id_lasso) == 0:
    Z_post = W[:, [0]]  # only d
else:
    Z_post = np.column_stack((W[:, [0]], W[:, id_lasso]))  # d + selected Zs

id_lasso

array([  0,  75,  86,  99, 147, 172, 182])

# Double-selection 

- Step 1: LASSO of X on Z
- Step 2: union of selected indices from y-LASSO and X-LASSO
- Step 3: Prepare X for post-double-LASSO SFA

In [34]:
# --- Step 1: LASSO of d on X (double-selection) ---
lasso_X = LassoCV(cv=5, fit_intercept=False, n_jobs=-1).fit(Z,X)
id_lasso_X = np.where(lasso_X.coef_ != 0)[0]
# --- Step 2: union of selected indices from y-LASSO and x-LASSO ---
id_yxlasso = np.union1d(id_lasso, id_lasso_X).astype(int)
# --- Step 3: Prepare X for post-double-LASSO SFA ---
if len(id_yxlasso) == 0:
    Z_Xlasso = W[:, [0]]  # only d
else:
    Z_Xlasso = np.column_stack((W[:, [0]], W[:, id_yxlasso]))

# --- Step 4: Fit SFA ---
sfa_Xlasso = SFA.SFA(y_sfa, Z_Xlasso)
sfa_Xlasso.optimize()

# --- Step 5: Extract key parameters (intercept, d-coef, sigma_v, sigma_u) ---
beta_hat_Xlasso = sfa_Xlasso.get_beta()
sigma_v_Xlasso = np.sqrt(sfa_Xlasso.get_sigmav2())
sigma_u_Xlasso = np.sqrt(sfa_Xlasso.get_sigmau2())


/home/zwei1/.local/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:1623: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [37]:
np.array([
    beta_hat_Xlasso[0],  # intercept
    beta_hat_Xlasso[1],  # coefficient on d
    sigma_v_Xlasso,      # noise SD
    sigma_u_Xlasso       # inefficiency SD
])

array([-9.69923413e-01,  2.68280170e-01,  5.80388902e-01,  3.12526349e-04])